<a href="https://colab.research.google.com/github/matteo-zoncheddu/AMD-Project-2025-26---Matteo-Zoncheddu/blob/main/AMD_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **AMD Project**
The task is to implement from scratch a system finding frequent itemsets (aka market-basket analysis) implementing at least one of the methods explained in the course. The detector can consider:
- **users** as items
- baskets as **articles**, described as **the sets of users who commented on them**.


### **Data Preparation**

In [ ]:
!pip install pandas numpy kagglehub

In [ ]:
# DATASET DOWNLOAD

# PERCENTAGE OF DATA USED FOR THE ALGORITHM
# CHOOSE A VALUE BETWEEN 1 AND 100 TO RUN THE ALGORITMH ON A SUBSET OF TOTAL BASKETS
DATA_PERCENTAGE = 100

# SUPER-GLOBAL VARIABLES
CHUNK_SIZE = 4000
SUPPORT_PERCENT = 1.7
SUPPORT_FLOOR = 30
RANDOMNESS = False

# Check and Cap on data percentage
DATA_PERCENTAGE = min(max(1, DATA_PERCENTAGE), 100)

# All libraries used in the project
import kagglehub
import os, csv
import shutil
import pandas as pd
import numpy as np
import itertools
import random
from itertools import combinations, islice
from collections import defaultdict

# Define the dataset handle and local destination
dataset_handle = "benjaminawd/new-york-times-articles-comments-2020"
local_data_path = 'data'
path = "data/nyt-comments-2020.csv"

# Only download if the data folder is missing
if not os.path.exists(local_data_path):
    print("Downloading dataset...")

    # Download the files and returns the path to a cache folder
    download_path = kagglehub.dataset_download(dataset_handle)

    # Move the files to data folder
    shutil.copytree(download_path, local_data_path)

    print(f"Dataset stored in: {local_data_path}")
else:
    print("Data folder already exists.")

In [ ]:
# PCY FUNCTIONS

def map_and_count_IDs(chunk, local_threshold):
    """
    Maps all items to small integers within the current chunk
    Counts all single item occurrences within the current chunk
    Returns:
        - local_item_to_int: local dictionary mapping items to {0,1,2,...}
        - local_frequent_items: set of locally frequent items
    """
    local_item_to_int = {}
    local_counts = []
    next_local_index = 0

    for basket in chunk:
        for user_id in basket:
            # Local mapping IDs to {0, 1, 2, ...}
            if user_id not in local_item_to_int:
                local_item_to_int[user_id] = next_local_index
                local_counts.append(0)
                next_local_index += 1

            idx = local_item_to_int[user_id]
            local_counts[idx] += 1

    # Extract locally frequent items
    local_frequent_items = {
        user_id for user_id, idx in local_item_to_int.items()
        if local_counts[idx] >= local_threshold
    }

    return local_item_to_int, local_frequent_items

def build_hash_table(chunk, local_frequent_items, hash_table_size=10000007):
    """
    Generates all pairs in each basket using only locally frequent items
    and hashes them into buckets.
    """
    hash_table = np.zeros(hash_table_size, dtype=np.uint32)

    print(f"Hashing pairs for {len(chunk)} baskets...")

    for basket in chunk:
        # Keep only locally frequent items
        filtered_items = [user_id for user_id in basket if user_id in local_frequent_items]

        if len(filtered_items) < 2:
            continue

        items = sorted(filtered_items)

        for i, j in itertools.combinations(items, 2):
            bucket_index = (hash(i) ^ hash(j)) % hash_table_size
            hash_table[bucket_index] += 1

    return hash_table

def generate_bitmap(hash_table, local_threshold):
    """
    Converts the integer hash table into a compact bitmap.
    """
    print(f"Generating bitmap with local threshold: {local_threshold}")

    # Boolean arrays are 1-byte per element in numpy, very memory efficient
    bitmap = hash_table >= local_threshold

    frequent_buckets = np.sum(bitmap)
    pruning_rate = (1 - (frequent_buckets / len(hash_table))) * 100

    print(f"Bitmap created!\nPruning rate: {pruning_rate:.2f}%")

    return bitmap

def get_pcy_triples(chunk, frequent_items, bitmap, bucket_size, local_threshold):
    """
    Pass 2 of PCY: Builds (i, j, count) triples for candidate pairs.
    """
    candidate_triples = {}
    print(f"Finding frequent itemsets...")

    for basket in chunk:
        # Only keep frequent users before doing combinations
        frequent_users = sorted([u for u in basket if u in frequent_items])

        for i, j in combinations(frequent_users, 2):
            # Use same hash logic as build_hash_table
            hash_val = (hash(i) ^ hash(j)) % bucket_size

            if bitmap[hash_val]:
                pair = (i, j)
                candidate_triples[pair] = candidate_triples.get(pair, 0) + 1

    return [(p[0], p[1], c) for p, c in candidate_triples.items() if c >= local_threshold]

### **SON Algorithm**

**Step 1: Set of All Local Candidates**

In [ ]:
#SON - STEP 1

# Storage for candidates
candidate_items = set() # Frequent local Singletons
candidate_pairs = set() # Frequent local Pairs
candidate_triples_set = set() # Frequent local Triples

total_baskets = 0
global_threshold = 0

# Chunking function
def get_complete_SON_chunks(comments_path, chunk_size, fraction, randomness):
    global total_baskets, global_threshold
    all_baskets_dict = {}

    reader = pd.read_csv(comments_path, usecols=['articleID', 'userID'], chunksize=200000)

    # Group by article
    for df_chunk in reader:
        for row in df_chunk.itertuples(index=False):
            if row.articleID not in all_baskets_dict:
                all_baskets_dict[row.articleID] = set()
            all_baskets_dict[row.articleID].add(row.userID)

    all_article_ids = list(all_baskets_dict.keys())
    num_to_keep = int(len(all_article_ids) * fraction)

    if randomness:
        # Random subsetting
        random.seed()
        selected_ids = random.sample(all_article_ids, num_to_keep)
    else:
        # Subsetting taking the first elements in order
        selected_ids = all_article_ids[:num_to_keep]

    # Create the final subsetted dictionary
    subsetted_baskets = {aid: all_baskets_dict[aid] for aid in selected_ids}

    # Cleanup memory of the large dictionary
    all_baskets_dict.clear()

    total_baskets = len(subsetted_baskets)

    # Calculate threshold based on the subset size
    global_threshold = max(SUPPORT_FLOOR, int(total_baskets * SUPPORT_PERCENT / 100))

    print(f"Final Basket Count: {total_baskets}")
    print(f"Global Threshold: {global_threshold}")

    # Yield chunks for SON Phase 1
    basket_list = list(subsetted_baskets.values())
    for i in range(0, total_baskets, chunk_size):
        yield basket_list[i:i + chunk_size]

def run_SON_phase_1():
    print(f"Data subsetted to {DATA_PERCENTAGE}% of total baskets.")

    # Initialize the generator
    basket_gen = get_complete_SON_chunks(
        path,
        chunk_size = CHUNK_SIZE,
        fraction = DATA_PERCENTAGE/100,
        randomness = RANDOMNESS
    )

    print("Running Phase 1 of SON...")

    # Main cycle
    for chunk_index, chunk in enumerate(basket_gen):
        print(f"\n\n--- Processing chunk {chunk_index + 1}:")

        # Calculate threshold for this chunk
        current_local_threshold = max(1, int(global_threshold * (len(chunk) / total_baskets)))

        # Local mapping and counting
        local_item_to_int, local_frequent_items = map_and_count_IDs(chunk, current_local_threshold)

        print(f"Unique items in current chunk: {len(local_item_to_int)}")

        candidate_items.update(local_frequent_items)
        print(f"Found {len(local_frequent_items)} frequent comments.")

        # Run PCY Between Step - Build Hash Table & Bitmap
        hash_table = build_hash_table(chunk, local_frequent_items)
        bitmap = generate_bitmap(hash_table, current_local_threshold)

        # Run PCY Step 2 - Finding sets of cardinality 2 and 3
        # Triples method chosen over triangular matrix
        local_triples = get_pcy_triples(
            chunk,
            local_frequent_items,
            bitmap,
            10000007,
            current_local_threshold
        )

        # Find local frequent sets with cardinality 2
        current_local_pairs = set()
        for i, j, count in local_triples:
            pair = tuple(sorted((i, j)))
            candidate_pairs.add(pair)
            current_local_pairs.add(pair)

        # Find local frequent sets with cardinality 3
        local_triple_counts = {}
        for basket in chunk:
            # Only keep items that are locally frequent
            frequent_in_basket = sorted([u for u in basket if u in local_frequent_items])
            if len(frequent_in_basket) < 3: continue

            for triple in combinations(frequent_in_basket, 3):
                # Monotonicity exploiting: card-3 set is candidate only if all its card-2 set are local frequent
                p1, p2, p3 = (triple[0], triple[1]), (triple[0], triple[2]), (triple[1], triple[2])
                if p1 in current_local_pairs and p2 in current_local_pairs and p3 in current_local_pairs:
                    local_triple_counts[triple] = local_triple_counts.get(triple, 0) + 1

        for triple, count in local_triple_counts.items():
            if count >= current_local_threshold:
                candidate_triples_set.add(triple)

    print(f"\nPhase 1 Complete!")
    print(f"Total Candidate Single Items: {len(candidate_items)}")
    print(f"Total Candidate Pairs: {len(candidate_pairs)}")
    print(f"Total Candidate Triples: {len(candidate_triples_set)}")

run_SON_phase_1()

**Step 2: Second Pass over the Full Dataset**

In [ ]:
# SON - STEP 2

# Global storage for the final results
final_frequent_pairs = {}
final_frequent_triples = {}

def run_SON_phase_2():
    global final_frequent_pairs, final_frequent_triples
    print("Running Phase 2 of SON...")

    # Prepare a numpy array and an index map for cardinalty-2 sets
    sorted_candidates = sorted(list(candidate_pairs))
    global_pair_counts = np.zeros(len(sorted_candidates), dtype=np.uint32)
    index_map = defaultdict(dict)
    for idx, (u1, u2) in enumerate(sorted_candidates):
        index_map[u1][u2] = idx

    # Prepare a numpy array and an index map for cardinalty-3 sets
    sorted_triples = sorted(list(candidate_triples_set))
    global_triple_counts = np.zeros(len(sorted_triples), dtype=np.uint32)
    triple_index_map = defaultdict(lambda: defaultdict(dict))
    for idx, (u1, u2, u3) in enumerate(sorted_triples):
        triple_index_map[u1][u2][u3] = idx

    # Re-initialize the generator to stream data again
    validation_gen = get_complete_SON_chunks(
        path,
        chunk_size = CHUNK_SIZE,
        fraction = DATA_PERCENTAGE / 100,
        randomness = RANDOMNESS
    )

    for chunk in validation_gen:
        for basket in chunk:
            # Filter for candidate items
            valid_items = sorted([u for u in basket if u in candidate_items])
            if len(valid_items) < 2: continue

            # Count cardinality-2 sets
            for pos, i in enumerate(valid_items):
                if i in index_map:
                    targets = index_map[i]
                    for j in valid_items[pos + 1:]:
                        if j in targets:
                            global_pair_counts[targets[j]] += 1

            # Count cardinality-3 sets
            if len(valid_items) >= 3:
                for i_pos, i in enumerate(valid_items):
                    if i in triple_index_map:
                        for j_pos, j in enumerate(valid_items[i_pos + 1:]):
                            if j in triple_index_map[i]:
                                targets_k = triple_index_map[i][j]
                                for k in valid_items[i_pos + j_pos + 2:]:
                                    if k in targets_k:
                                        global_triple_counts[targets_k[k]] += 1

    # Final filter for card-2 sets
    final_frequent_pairs = {
        sorted_candidates[idx]: global_pair_counts[idx]
        for idx in np.where(global_pair_counts >= global_threshold)[0]
    }

    # Final filter for card-3 sets
    final_frequent_triples = {
        sorted_triples[idx]: global_triple_counts[idx]
        for idx in np.where(global_triple_counts >= global_threshold)[0]
    }

    print(f"Algorithm completed!")
    print(f"Found {len(final_frequent_pairs)} frequent pairs.")
    print(f"Found {len(final_frequent_triples)} frequent triples.")

run_SON_phase_2()

### **Final Output**

In [ ]:
# EXPORT RESULTS

output_file = "Frequent_Itemsets.csv"

# Combine results
all_frequent_results = []

# Add cardinality-2 sets
for (u1, u2), count in final_frequent_pairs.items():
    all_frequent_results.append({
        'Items': f"{u1}, {u2}",
        'Size': 2,
        'Support': count
    })

# Add cardinality-3 sets
for (u1, u2, u3), count in final_frequent_triples.items():
    all_frequent_results.append({
        'Items': f"{u1}, {u2}, {u3}",
        'Size': 3,
        'Support': count
    })

# Sort the results by count so the most frequent are first
sorted_results = sorted(
    all_frequent_results,
    key=lambda x: x['Support'],
    reverse=True
)

print(f"Saving results to {output_file}...")

try:
    with open(output_file, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)

        # Write the header
        writer.writerow(['Items', 'Cardinality', 'Global Support Count'])

        # Write the itemsets
        for row in sorted_results:
            writer.writerow([row['Items'], row['Size'], row['Support']])

    print(f"Successfully saved {len(sorted_results)} frequent itemsets to: \n{os.path.abspath(output_file)}")

except Exception as e:
    print(f"An error occurred while saving: {e}")